# Kiva Lending Patterns - Exploratory Data Analysis

Loads the cleaned dataset and produces all exploratory visualizations.

---

## Task 2: Exploratory Visualizations

Nine charts covering all proposed visualization ideas. Three will be selected as assessed visualizations for the final submission. All charts load the cleaned dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

CLEANED_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
dfc = pd.read_csv(CLEANED_PATH, index_col=0)
print(f"Cleaned dataset loaded: {dfc.shape[0]:,} rows, {dfc.shape[1]} columns")

### Chart 1 - Loan Count by Sector (horizontal bar chart)

**Objective:** Show which sectors account for the most Kiva loan records.  
**What to look for:** Dominant sectors, long tail of small sectors.  
**Limitation:** Counts reflect lending activity only -- not monetary volume or borrower outcomes.

In [ ]:
sector_counts = dfc['sector'].value_counts().reset_index()
sector_counts.columns = ['sector', 'count']
sector_counts['pct'] = (sector_counts['count'] / len(dfc) * 100).round(1)

fig1 = px.bar(
    sector_counts,
    x='count',
    y='sector',
    orientation='h',
    text=sector_counts['pct'].map('{:.1f}%'.format),
    title='Kiva Loans by Sector',
    labels={'count': 'Number of Loans', 'sector': 'Sector'}
)
fig1.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)
fig1.show()

### Chart 2 - Loan Count by Country (horizontal bar chart, top 25)

**Objective:** Identify which countries have the most Kiva loan records.  
**What to look for:** Geographic concentration, gap between the top country and the rest.  
**Limitation:** Country population and economic size affect counts; more populous countries may simply have more borrowers.

In [ ]:
country_counts = dfc['country'].value_counts().head(25).reset_index()
country_counts.columns = ['country', 'count']
country_counts['pct'] = (country_counts['count'] / len(dfc) * 100).round(1)

fig2 = px.bar(
    country_counts,
    x='count',
    y='country',
    orientation='h',
    text=country_counts['pct'].map('{:.1f}%'.format),
    title='Top 25 Countries by Number of Kiva Loans',
    labels={'count': 'Number of Loans', 'country': 'Country'}
)
fig2.update_layout(yaxis={'categoryorder': 'total ascending'})
fig2.show()

### Chart 3 - Loan Amount Distribution by Sector (box plot)

**Objective:** Compare the spread and central tendency of loan amounts across sectors.  
**What to look for:** Sectors with unusually large or small loans, wide vs. narrow distributions, outliers.  
**Limitation:** Box plots can hide the shape of the distribution within the box. Very skewed distributions may appear deceptively compact.

In [ ]:
sector_order_3 = (
    dfc.groupby('sector')['loan_amount']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig3 = px.box(
    dfc,
    x='sector',
    y='loan_amount',
    title='Loan Amount Distribution by Sector (USD)',
    labels={'loan_amount': 'Loan Amount (USD)', 'sector': 'Sector'},
    category_orders={'sector': sector_order_3}
)
fig3.update_layout(xaxis_tickangle=-45, xaxis_title=None)
fig3.show()

### Chart 4 - Repayment Interval Distribution by Sector (100% stacked bar)

**Objective:** Show what proportion of loans in each sector use each repayment interval.  
**What to look for:** Sectors dominated by a single interval vs. mixed distributions.  
**Limitation:** Proportions only; does not show repayment success. Sample sizes are included in sector labels.

In [ ]:
repay_counts = (
    dfc.groupby(['sector', 'repayment_interval'])
    .size()
    .reset_index(name='count')
)
sector_totals = dfc.groupby('sector').size().reset_index(name='total')
repay_counts = repay_counts.merge(sector_totals, on='sector')
repay_counts['pct'] = repay_counts['count'] / repay_counts['total'] * 100
repay_counts['sector_label'] = repay_counts.apply(
    lambda r: f"{r['sector']} (n={r['total']:,})", axis=1
)

sector_order_4 = (
    sector_totals.sort_values('total', ascending=False)['sector'].tolist()
)
label_order_4 = [
    repay_counts.loc[repay_counts['sector'] == s, 'sector_label'].iloc[0]
    for s in sector_order_4
]

fig4 = px.bar(
    repay_counts,
    x='pct',
    y='sector_label',
    color='repayment_interval',
    orientation='h',
    title='Repayment Interval Distribution by Sector (% of loans)',
    labels={
        'pct': '% of Loans',
        'sector_label': 'Sector',
        'repayment_interval': 'Interval'
    },
    category_orders={'sector_label': label_order_4}
)
fig4.update_layout(xaxis_ticksuffix='%')
fig4.show()

### Chart 5 - Loan Duration by Sector (box plot of `term_in_months`)

**Objective:** Compare loan durations across sectors.  
**What to look for:** Sectors with longer or shorter typical durations, unusually long outliers.  
**Limitation:** Duration describes loan structure, not repayment success.

In [ ]:
term_data = dfc[dfc['term_in_months'].notna()]
sector_order_5 = (
    term_data.groupby('sector')['term_in_months']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig5 = px.box(
    term_data,
    x='sector',
    y='term_in_months',
    title='Loan Duration by Sector (months)',
    labels={'term_in_months': 'Duration (months)', 'sector': 'Sector'},
    category_orders={'sector': sector_order_5}
)
fig5.update_layout(xaxis_tickangle=-45, xaxis_title=None)
fig5.show()

### Chart 6 - Funding Gap by Sector (box plot)

**Objective:** Show how the difference between requested and funded amounts varies across sectors.  
**What to look for:** Sectors where loans are typically fully funded (gap near 0) vs. sectors with persistent shortfalls.  
**Note:** A negative gap means the funded amount exceeded the requested amount, which is valid per the assignment context.  
**Limitation:** Values beyond the 99th percentile are excluded for readability; they are not dropped from the dataset.

In [ ]:
p99 = dfc['funding_gap'].quantile(0.99)
gap_data = dfc[dfc['funding_gap'] <= p99].copy()
print(f"Records included (up to 99th percentile, gap <= {p99:.0f}): {len(gap_data):,}")

sector_order_6 = (
    gap_data.groupby('sector')['funding_gap']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig6 = px.box(
    gap_data,
    x='sector',
    y='funding_gap',
    title='Funding Gap by Sector (loan_amount minus funded_amount, capped at 99th percentile)',
    labels={'funding_gap': 'Funding Gap (USD)', 'sector': 'Sector'},
    category_orders={'sector': sector_order_6}
)
fig6.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Fully funded')
fig6.update_layout(xaxis_tickangle=-45, xaxis_title=None)
fig6.show()

### Chart 7 - Threshold-Based Hierarchical Treemap

**Objective:** Show the composition of Kiva loans by sector and country in a single hierarchical chart.  
**Rule:** At each level (sector and country within sector), categories are added from largest to smallest until the remaining ungrouped share drops to 10% or below. The rest is grouped into 'Other'.  
**What to look for:** Dominant sector-country combinations, overall share vs. share within a sector.  
**Limitation:** Treemaps are poor for precise size comparison between non-adjacent rectangles. Bar charts remain clearer for ranking.

In [ ]:
def apply_threshold(count_series, total, other_label, threshold_pct=10.0):
    """Return an ordered dict of {label: count} including an Other group if needed."""
    displayed = {}
    remaining = total
    for label, count in count_series.sort_values(ascending=False).items():
        remaining -= count
        displayed[label] = count
        if remaining / total * 100 <= threshold_pct:
            break
    other_count = total - sum(displayed.values())
    if other_count > 0:
        displayed[other_label] = other_count
    return displayed


total_loans = len(dfc)
sector_counts_full = dfc['sector'].value_counts()
sector_displayed = apply_threshold(sector_counts_full, total_loans, 'Other sectors')

rows = [{'id': 'All Loans', 'label': 'All Loans', 'parent': '',
         'value': total_loans, 'overall_pct': 100.0, 'parent_pct': 100.0}]

other_sector_names = [s for s in sector_displayed if s != 'Other sectors']

for sector_label, sector_count in sector_displayed.items():
    rows.append({
        'id': sector_label,
        'label': sector_label,
        'parent': 'All Loans',
        'value': sector_count,
        'overall_pct': round(sector_count / total_loans * 100, 2),
        'parent_pct': round(sector_count / total_loans * 100, 2)
    })

    if sector_label == 'Other sectors':
        subset = dfc[~dfc['sector'].isin(other_sector_names)]
    else:
        subset = dfc[dfc['sector'] == sector_label]

    country_counts = subset['country'].value_counts()
    country_displayed = apply_threshold(country_counts, sector_count, 'Other countries')

    for country_label, country_count in country_displayed.items():
        rows.append({
            'id': f'{sector_label}|{country_label}',
            'label': country_label,
            'parent': sector_label,
            'value': country_count,
            'overall_pct': round(country_count / total_loans * 100, 2),
            'parent_pct': round(country_count / sector_count * 100, 2)
        })

treemap_df = pd.DataFrame(rows)

# Validation
sector_total_check = treemap_df[treemap_df['parent'] == 'All Loans']['value'].sum()
print(f"Sectors shown individually: {len(treemap_df[treemap_df['parent'] == 'All Loans']) - 1} + Other")
print(f"Treemap sector total matches dataset: {sector_total_check == total_loans}")
other_sector_pct = treemap_df.loc[treemap_df['id'] == 'Other sectors', 'overall_pct'].values
if len(other_sector_pct):
    print(f"Other sectors share: {other_sector_pct[0]:.1f}% (target: <= 10%)")

In [ ]:
fig7 = px.treemap(
    treemap_df,
    ids='id',
    names='label',
    parents='parent',
    values='value',
    custom_data=['overall_pct', 'parent_pct'],
    title='Kiva Loan Distribution Across Sectors and Countries'
)
fig7.update_traces(
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Loans: %{value:,}<br>'
        'Overall share: %{customdata[0]:.1f}%<br>'
        'Parent share: %{customdata[1]:.1f}%'
        '<extra></extra>'
    ),
    textinfo='label+value+percent root'
)
fig7.show()

### Chart 8 - Median Loan Amount by Country (horizontal bar chart, top 20)

**Objective:** Identify which countries have the highest typical loan amounts, independent of how many loans they have.  
**Filter:** Only countries with at least 500 records are included to avoid median estimates based on very few loans.  
**What to look for:** High-median countries may reflect stronger economies, different Kiva program types, or specific sectors that dominate in those countries.

In [ ]:
country_stats = (
    dfc.groupby('country')['loan_amount']
    .agg(['median', 'count'])
    .reset_index()
)
country_stats.columns = ['country', 'median_loan', 'count']
top_by_median = country_stats[country_stats['count'] >= 500].nlargest(20, 'median_loan')

fig8 = px.bar(
    top_by_median,
    x='median_loan',
    y='country',
    orientation='h',
    text=top_by_median['count'].map(lambda n: f'n={n:,}'),
    title='Top 20 Countries by Median Loan Amount (min 500 loans)',
    labels={'median_loan': 'Median Loan Amount (USD)', 'country': 'Country'}
)
fig8.update_layout(yaxis={'categoryorder': 'total ascending'})
fig8.show()

### Chart 9 - Mean Lender Count vs Median Loan Amount by Sector (scatter)

**Objective:** Explore whether larger loans attract more lenders, and whether this pattern differs by sector.  
**Aggregation:** Each point represents one sector. Bubble size = total loan count in that sector.  
**What to look for:** A positive trend would suggest larger loans mobilize more lenders. Outlier sectors that attract few lenders despite large loans (or many lenders despite small loans) are interesting.  
**Limitation:** Sector-level aggregation hides the within-sector distribution.

In [ ]:
sector_agg = (
    dfc.groupby('sector')
    .agg(
        median_loan_amount=('loan_amount', 'median'),
        mean_lender_count=('lender_count', 'mean'),
        loan_count=('loan_amount', 'count')
    )
    .reset_index()
)

fig9 = px.scatter(
    sector_agg,
    x='median_loan_amount',
    y='mean_lender_count',
    size='loan_count',
    text='sector',
    color='sector',
    title='Mean Lender Count vs Median Loan Amount by Sector (bubble size = loan count)',
    labels={
        'median_loan_amount': 'Median Loan Amount (USD)',
        'mean_lender_count': 'Mean Lender Count',
        'loan_count': 'Number of Loans'
    }
)
fig9.update_traces(textposition='top center')
fig9.update_layout(showlegend=False)
fig9.show()